# HAM10000 Preprocessing (Stage 1 only)

This notebook prepares HAM10000 train/val/test splits using **patient-level splitting** and saves CSVs for training in `model.ipynb`.

In [1]:
import os
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit

DATA_ROOT = Path('data/HAM10000')
META_CSV = DATA_ROOT / 'HAM10000_metadata.csv'
IMG_DIRS = [
    DATA_ROOT / 'HAM10000_images_part_1',
    DATA_ROOT / 'HAM10000_images_part_2',
]
SEED = 42
np.random.seed(SEED)

HAM_CLASSES = ['akiec', 'bcc', 'bkl', 'df', 'mel', 'nv', 'vasc']
LABEL_MAP = {c: i for i, c in enumerate(HAM_CLASSES)}
print('LABEL_MAP:', LABEL_MAP)

LABEL_MAP: {'akiec': 0, 'bcc': 1, 'bkl': 2, 'df': 3, 'mel': 4, 'nv': 5, 'vasc': 6}


In [2]:
def resolve_image_path(image_id: str) -> str | None:
    for d in IMG_DIRS:
        p = d / f'{image_id}.jpg'
        if p.exists():
            return str(p)
    return None


df = pd.read_csv(META_CSV)
required_cols = ['image_id', 'dx', 'lesion_id']
for col in required_cols:
    if col not in df.columns:
        raise ValueError(f'Missing required metadata column: {col}')

if 'patient_id' not in df.columns:
    warnings.warn('patient_id not found; using lesion_id as patient_id proxy.')
    df['patient_id'] = df['lesion_id']

before = len(df)
df = df[df['dx'].isin(HAM_CLASSES)].copy()
print(f'Filtered labels: {before} -> {len(df)} rows')

df['image_path'] = df['image_id'].astype(str).map(resolve_image_path)
missing = df['image_path'].isna().sum()
if missing:
    warnings.warn(f'{missing} images missing on disk; dropping them.')
df = df.dropna(subset=['image_path']).copy()

df['label'] = df['dx'].map(LABEL_MAP).astype(int)

print('Final dataframe shape:', df.shape)
print(df['dx'].value_counts())
print(df.head(3))

C:\Users\prakh\AppData\Local\Temp\ipykernel_10564\3269781332.py:16: UserWarning: patient_id not found; using lesion_id as patient_id proxy.
  warnings.warn('patient_id not found; using lesion_id as patient_id proxy.')


Filtered labels: 10015 -> 10015 rows
Final dataframe shape: (10015, 10)
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115
Name: count, dtype: int64
     lesion_id      image_id   dx dx_type   age   sex localization  \
0  HAM_0000118  ISIC_0027419  bkl   histo  80.0  male        scalp   
1  HAM_0000118  ISIC_0025030  bkl   histo  80.0  male        scalp   
2  HAM_0002730  ISIC_0026769  bkl   histo  80.0  male        scalp   

    patient_id                                         image_path  label  
0  HAM_0000118  data\HAM10000\HAM10000_images_part_1\ISIC_0027...      2  
1  HAM_0000118  data\HAM10000\HAM10000_images_part_1\ISIC_0025...      2  
2  HAM_0002730  data\HAM10000\HAM10000_images_part_1\ISIC_0026...      2  


In [3]:
def patient_level_split(
    in_df: pd.DataFrame,
    patient_col: str = 'patient_id',
    label_col: str = 'dx',
    val_ratio: float = 0.15,
    test_ratio: float = 0.15,
    seed: int = 42,
):
    groups = in_df.groupby(patient_col)
    patients = []
    patient_labels = []

    for pid, g in groups:
        patients.append(pid)
        patient_labels.append(g[label_col].mode(dropna=False).iloc[0])

    patients = np.array(patients)
    patient_labels = np.array(patient_labels)

    idx = np.arange(len(patients))
    sss_test = StratifiedShuffleSplit(n_splits=1, test_size=test_ratio, random_state=seed)
    train_val_idx, test_idx = next(sss_test.split(idx, patient_labels))

    train_val_patients = patients[train_val_idx]
    train_val_labels = patient_labels[train_val_idx]

    val_adj = val_ratio / (1.0 - test_ratio)
    sss_val = StratifiedShuffleSplit(n_splits=1, test_size=val_adj, random_state=seed)
    tr_idx_local, va_idx_local = next(sss_val.split(np.arange(len(train_val_patients)), train_val_labels))

    train_patients = set(train_val_patients[tr_idx_local])
    val_patients = set(train_val_patients[va_idx_local])
    test_patients = set(patients[test_idx])

    train_df = in_df[in_df[patient_col].isin(train_patients)].copy()
    val_df = in_df[in_df[patient_col].isin(val_patients)].copy()
    test_df = in_df[in_df[patient_col].isin(test_patients)].copy()

    return train_df, val_df, test_df


train_df, val_df, test_df = patient_level_split(df)
print('Train:', len(train_df), ' Val:', len(val_df), ' Test:', len(test_df))

Train: 7054  Val: 1464  Test: 1497


In [4]:
train_p = set(train_df['patient_id'].astype(str))
val_p = set(val_df['patient_id'].astype(str))
test_p = set(test_df['patient_id'].astype(str))

assert len(train_p & val_p) == 0, 'Leakage: train/val patient overlap'
assert len(train_p & test_p) == 0, 'Leakage: train/test patient overlap'
assert len(val_p & test_p) == 0, 'Leakage: val/test patient overlap'

print('Patient-level split checks passed.')
print('Class counts (train):')
print(train_df['dx'].value_counts())

Patient-level split checks passed.
Class counts (train):
dx
nv       4730
mel       777
bkl       775
bcc       365
akiec     233
vasc       98
df         76
Name: count, dtype: int64


In [5]:
out_cols = ['image_id', 'image_path', 'dx', 'label', 'lesion_id', 'patient_id']

train_path = DATA_ROOT / 'ham_train.csv'
val_path = DATA_ROOT / 'ham_val.csv'
test_path = DATA_ROOT / 'ham_test.csv'

train_df[out_cols].to_csv(train_path, index=False)
val_df[out_cols].to_csv(val_path, index=False)
test_df[out_cols].to_csv(test_path, index=False)

print('Saved:')
print(' -', train_path, len(train_df))
print(' -', val_path, len(val_df))
print(' -', test_path, len(test_df))

Saved:
 - data\HAM10000\ham_train.csv 7054
 - data\HAM10000\ham_val.csv 1464
 - data\HAM10000\ham_test.csv 1497
